In [1]:
import csv
import json
import re

from scielo_scholarly_data import standardizer
from scielo_scholarly_data.standardizer import ImpossibleConvertionToIntError, InvalidRomanNumeralError

In [18]:
path_wos_source = '/home/rafaeljpd/Downloads/WoS_RegsC_TitCorr_TB.txt'
path_wos_source_missing = '/home/rafaeljpd/Downloads/WoS_RegsC_TitCorr_TB.missing.txt'
save_missing = False

path_wos_output = '/home/rafaeljpd/Downloads/output.wos.json'
path_wos_output_missing = '/home/rafaeljpd/Downloads/output.wos.missing.json'

wos_source_fieldnames = [
    'id',
    'citation_count',
    'cited_doiset',
    'cited_journal',
    'cited_year',
    'cited_vol',
]

regex_volume = r'^(?P<posfix>[v|V])(?P<volume>\d*)$'

In [3]:
def fix_volume(text):
	if text.isdigit():
		return text

	m = re.match(regex_volume, text)
	if m:
		return m.groupdict().get('volume')

	try:
		return standardizer.issue_volume(text)
	except (ImpossibleConvertionToIntError, InvalidRomanNumeralError):
		return standardizer.issue_volume(text, force_integer=False)


def standardize_data(data):        
    if 'cited_doiset' in data or 'cited_doi' in data:
        cited_doiset = set()
        if data['cited_doiset'] is None:
            print(data)
        else:
            for d in data['cited_doiset'].split(' '):
                doi_stz = standardizer.document_doi(d, return_mode='path')
                if not isinstance(doi_stz, dict):
                    cited_doiset.add(doi_stz)
        if len(cited_doiset) > 0:
            data['cited_doiset'] = '#'.join(cited_doiset)

    if 'cited_vol' in data:
        fixed_vol = fix_volume(data['cited_vol']).strip()
        data['cited_vol'] = fixed_vol

    if 'cited_journal' in data:
        data['cited_journal'] = standardizer.journal_title_for_deduplication(data['cited_journal'].strip()).upper()

    if 'cited_year' in data:
        data['cited_year'] = data['cited_year'].strip()

    if 'cited_doiset' in data:
        data['cited_doiset'] = data['cited_doiset'].strip()


def genkey(data):
	standardize_data(data)
        
	return '|'.join([
        data['id'],
        data['citation_count'],
        data['cited_doiset'],
        data['cited_journal'],
        data['cited_year'],
        data['cited_vol']
    ])

In [4]:
wos_output = []

with open(path_wos_output) as fin:
    for line in fin:
        jline = json.loads(line)
        wos_output.append(jline)

wos_output

[{'citation_count': '4615',
  'cited_doiset': '',
  'cited_issnl': '0021-9258',
  'cited_journal': 'J BIOL CHEM',
  'cited_vol': '193',
  'cited_year': '1951',
  'id': '999995384',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '4174',
  'cited_doiset': '',
  'cited_issnl': '0003-2697',
  'cited_journal': 'ANAL BIOCHEM',
  'cited_vol': '72',
  'cited_year': '1976',
  'id': '999995825',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '3406',
  'cited_doiset': '',
  'cited_issnl': '0021-9258',
  'cited_journal': 'J BIOL CHEM',
  'cited_vol': '276',
  'cited_year': '2001',
  'id': '999996593',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '3388',
  'cited_doiset': '',
  'cited_issnl': '0021-9258',
  'cited_journal': 'J BIOL CHEM',
  'cited_vol': '271',
  'cited_year': '1996',
  'id': '999996611',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '3367',
  'cited_doiset': '',
  'cited_issnl': '0021-9258',
  'cited_journal': 'J BIOL CHEM'

In [5]:
wos_output_missing = []

with open(path_wos_output_missing) as fin:
    for line in fin:
        jline = json.loads(line)
        wos_output_missing.append(jline)

wos_output_missing

[{'citation_count': '1',
  'cited_doiset': '',
  'cited_journal': 'NOMENCLATOR HELICEOR',
  'cited_vol': '',
  'cited_year': '1881',
  'id': '999999998',
  'result_code': 70},
 {'citation_count': '1',
  'cited_doiset': '',
  'cited_journal': 'OLD WORLD CERAMBYCID',
  'cited_vol': '',
  'cited_year': '2018',
  'id': '999999998',
  'result_code': 70},
 {'citation_count': '1',
  'cited_doiset': '',
  'cited_journal': 'NOTAS ACERCA LOGICA',
  'cited_vol': '52',
  'cited_year': '1995',
  'id': '999999998',
  'result_code': 70},
 {'citation_count': '1',
  'cited_doiset': '',
  'cited_journal': 'OFF FURN WORLD MARK',
  'cited_vol': '',
  'cited_year': '2015',
  'id': '999999998',
  'result_code': 70},
 {'citation_count': '1',
  'cited_doiset': '',
  'cited_journal': 'NUTR EXPL COM DIET P',
  'cited_vol': '',
  'cited_year': '2017',
  'id': '999999998',
  'result_code': 70},
 {'citation_count': '1',
  'cited_doiset': '',
  'cited_issnl': '0304-3940',
  'cited_journal': 'NEUROSCI LETT',
  'cite

In [6]:
wos_source = []

with open(path_wos_source, errors='ignore') as fin:
    cr = csv.DictReader(fin, fieldnames=wos_source_fieldnames, delimiter='|', restval='', escapechar='\\', quoting=csv.QUOTE_NONE)
    for row in cr:
        standardize_data(row)
        wos_source.append(row)

wos_source

[{'id': '999995384',
  'citation_count': '4615',
  'cited_doiset': '',
  'cited_journal': 'J BIOL CHEM',
  'cited_year': '1951',
  'cited_vol': '193'},
 {'id': '999995825',
  'citation_count': '4174',
  'cited_doiset': '',
  'cited_journal': 'ANAL BIOCHEM',
  'cited_year': '1976',
  'cited_vol': '72'},
 {'id': '999996593',
  'citation_count': '3406',
  'cited_doiset': '',
  'cited_journal': 'J BIOL CHEM',
  'cited_year': '2001',
  'cited_vol': '276'},
 {'id': '999996611',
  'citation_count': '3388',
  'cited_doiset': '',
  'cited_journal': 'J BIOL CHEM',
  'cited_year': '1996',
  'cited_vol': '271'},
 {'id': '999996632',
  'citation_count': '3367',
  'cited_doiset': '',
  'cited_journal': 'J BIOL CHEM',
  'cited_year': '1994',
  'cited_vol': '269'},
 {'id': '999996662',
  'citation_count': '3337',
  'cited_doiset': '',
  'cited_journal': 'J BIOL CHEM',
  'cited_year': '2000',
  'cited_vol': '275'},
 {'id': '999996700',
  'citation_count': '3299',
  'cited_doiset': '',
  'cited_journal'

In [7]:
len(wos_source), len(wos_output), len(wos_output_missing)

(11268288, 11161646, 123318)

In [8]:
wos_source[2974958]

{'id': '999999997',
 'citation_count': '2',
 'cited_doiset': '10.1016/J.IJCARD.2011.04.021',
 'cited_journal': 'INT J CARDIOL',
 'cited_year': '2012',
 'cited_vol': '161'}

In [9]:
wos_output[2974958]

{'citation_count': '2',
 'cited_doiset': '10.1016/J.IJCARD.2011.04.021',
 'cited_issnl': '0167-5273',
 'cited_journal': 'INT J CARDIOL',
 'cited_vol': '161',
 'cited_year': '2012',
 'id': '999999997',
 'issnls_size': 1,
 'result_code': 0}

In [10]:
wos_output_missing[0]

{'citation_count': '1',
 'cited_doiset': '',
 'cited_journal': 'NOMENCLATOR HELICEOR',
 'cited_vol': '',
 'cited_year': '1881',
 'id': '999999998',
 'result_code': 70}

In [11]:
source_key_to_line_number = {}
for ind, s in enumerate(wos_source):
    k = genkey(s)
    if k not in source_key_to_line_number:
        source_key_to_line_number[k] = []
    source_key_to_line_number[k].append(ind)

In [12]:
output_key_to_line_number = {}
for ind, o in enumerate(wos_output):
    k = genkey(o)
    if k not in output_key_to_line_number:
        output_key_to_line_number[k] = []
    output_key_to_line_number[k].append(ind)

In [13]:
output_missing_key_to_line_number = {}
for ind, om in enumerate(wos_output_missing):
    k = genkey(om)
    if k not in output_missing_key_to_line_number:
        output_missing_key_to_line_number[k] = []
    output_missing_key_to_line_number[k].append(ind)

In [14]:
for k, v in source_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

999999987|12||REVISTA UNIVERSIDADE RURAL SERIE CIENCIAS DA VIDA|2002|22 [284343, 284344, 284345]


In [15]:
for k, v in output_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

999999989|10|10.1175/1520-0493(2001)129|MON WEATHER REV|2001|129 [330583, 330584, 330585]


In [16]:
for k, v in output_missing_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

In [17]:
k1 = '999999989|10|10.1175/1520-0493(2001)129|MON WEATHER REV|2001|129'

for l in source_key_to_line_number[k1]:
    print(wos_source[l])

for k in output_key_to_line_number:
    if '|THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH' in k:
        print(k)

{'id': '999999989', 'citation_count': '10', 'cited_doiset': '10.1175/1520-0493(2001)129', 'cited_journal': 'MON WEATHER REV', 'cited_year': '2001', 'cited_vol': '129'}
{'id': '999999989', 'citation_count': '10', 'cited_doiset': '10.1175/1520-0493(2001)129', 'cited_journal': 'MON WEATHER REV', 'cited_year': '2001', 'cited_vol': '129'}
{'id': '999999989', 'citation_count': '10', 'cited_doiset': '10.1175/1520-0493(2001)129', 'cited_journal': 'MON WEATHER REV', 'cited_year': '2001', 'cited_vol': '129'}


In [19]:
def extract_value(data):
    return '|'.join([
        data.get('cited_issnl', ''),
        str(data.get('result_code', '')),
        str(data.get('issnls_size', '')),
        data.get('title_year_volume_key', '')
    ]), len(data.get('cited_issnl', '')) == 9

def find_best_result(chave, output_keys_1, output_keys_2={}):
    o1_lines = output_keys_1.get(chave, [])
    o2_lines = output_keys_2.get(chave, [])

    outs_with_issnl = set()
    outs = set()

    for ol in o1_lines:
        ol_v, has_issnl = extract_value(wos_output[ol])
        outs.add(ol_v)
        if has_issnl:
            outs_with_issnl.add(ol_v)

    for ol in o2_lines:
        ol_v, has_issnl = extract_value(wos_output_missing[ol])
        outs.add(ol_v)
        if has_issnl:
            outs_with_issnl.add(ol_v)

    # espera-se que todos os ISSNLs atribuídos para uma chave sejam os mesmos
    # então, pode-se retornar o primeiro
    if outs_with_issnl:
        return outs_with_issnl.pop(), True

    # espera-se que todos os erros identificados para uma chave sejam parecidas
    # então, pode-se retornar o primeiro
    try:
        return outs.pop(), True
    except:
        return chave, False

source_enriched = {}

null_value = '|'.join(['', '', '', ''])
missing_keys = set()

for k in source_key_to_line_number:
    v, success = find_best_result(k, output_key_to_line_number, output_missing_key_to_line_number)

    if not success:
        missing_keys.add(v)
    else:
        lines = source_key_to_line_number[k]    
        for l in lines:
            if len(v) > 0:
                source_enriched[l] = f'{k}|{v}'
            else:
                source_enriched[l] = f'{k}|{null_value}'

if save_missing:
    with open(path_wos_source_missing, 'w') as fout:
        for i in missing_keys:
            fout.write(i + '\n')

In [20]:
missing_keys

set()

In [21]:
with open('/home/rafaeljpd/Downloads/output.wos.fixed.csv', 'w') as fout:
    for k in sorted(source_enriched):
        fout.write(source_enriched[k] + '\n')

In [22]:
def to_json(data):
    return json.dumps(data, default=lambda o: o.__dict__, sort_keys=True, ensure_ascii=False)

def load_citation_from_str(str_data):
    data = {}

    els = str_data.split('|')

    if len(els) != 10:
        print(str_data)

    data['id'] = els[0].strip()
    data['citation_count'] = els[1].strip()
    data['cited_doiset'] = els[2].strip()
    data['cited_journal'] = els[3].strip()
    data['cited_year'] = els[4].strip()
    data['cited_vol'] = els[5].strip()
    
    if '<<<PROCESS>>>' not in str_data:
        data['cited_issnl'] = els[6].strip()
        try:
            data['result_code'] = int(els[7].strip())
        except ValueError:
            data['result_code'] = ''

        try:    
            data['issnls_size'] = int(els[8].strip())
        except ValueError:
            data['issnls_size'] = ''

        data['title_year_volume_key'] = els[9].strip()

    return data
    
with open('/home/rafaeljpd/Downloads/output.wos.fixed.json', 'w') as fout:
    for k in sorted(source_enriched):
        v = load_citation_from_str(source_enriched[k])
        fout.write(to_json(v) + '\n')

In [24]:
val = {}
with open('/home/rafaeljpd/Downloads/output.wos.fixed.json') as fin:
    for line in fin:
        jline = json.loads(line)
        rc = jline['result_code']
        if rc not in val:
            val[rc] = 0
        val[rc] += 1

TypeError: object of type 'int' has no len()

In [26]:
for k in sorted(val):
    print(k, val[k])

print(sum(val.values()))

0 7054182
1 928072
2 23647
3 5815
4 9080
11 253030
12 4569
13 20167
14 28992
70 2226999
80 917
81 54113
500 185903
529 12154
549 432
600 424655
619 25968
629 210
639 1267
649 8116
11268288
